# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n{metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their details
print('Record sets in the dataset:')
for rs in metadata.record_sets:
    print(f"- Name: {rs.name} | @id: {rs.id}")
    print('  Fields:')
    for field in rs.fields:
        print(f"    - {field.name} | @id: {field.id} | dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, we'll extract all available record sets
record_sets_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    # Load records for each record set
    recs = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(recs)
    dataframes[record_set_id] = df

if record_sets_ids:
    main_rs_id = record_sets_ids[0]
    print(f"First record set: {main_rs_id}")
    print("Fields/Columns:", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print('No record sets found in this dataset.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, pick a numeric field. Inspect the DataFrame columns for likely candidates, for example, 'Molecular Weight' or 'Coverage Percent'.
import numpy as np

# Attempt to auto-detect a numeric field
df = dataframes[main_rs_id]
numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int] or df[col].apply(lambda x: isinstance(x, (int,float))).all()]
if len(numeric_candidates) == 0:
    # Try to convert likely numeric columns
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            if df[col].dtype in [np.float64, np.int64]:
                numeric_candidates.append(col)
        except:
            pass

if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    numeric_field = df.columns[0]
    print(f"Warning: No obvious numeric field found, proceeding with '{numeric_field}'")

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a field if present (e.g., group by 'Sample' if exists)
group_field = None
preferred_groups = ['Sample', 'sample', 'Condition','condition','Description','description','Group','group','Protein','protein']
for col in df.columns:
    if col in preferred_groups:
        group_field = col
        break

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the selected numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field], bins=30, kde=True)
plt.xlabel(numeric_field)
plt.title(f'Distribution of {numeric_field}')
plt.show()

# If group_field exists, boxplot numeric value by group
if group_field and group_field in df.columns:
    plt.figure(figsize=(10, 4))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.xticks(rotation=45, ha='right')
    plt.title(f'{numeric_field} by {group_field}')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

<!-- Add your summary here. For example: -->
In this notebook, we loaded and explored the FAIR²-compliant mass spectrometry dataset of proteins isolated from human mast cell derived extracellular vesicles. We listed all available record sets and fields, loaded the records into DataFrames, and performed initial exploratory analysis including filtering and normalization on a numeric field. Visualizations revealed the field's distribution and, where possible, its breakdown by groups (such as sample or condition). This process demonstrates a reproducible approach to inspecting rich protein/proteomics datasets described in Croissant format using `mlcroissant`.